# Moment generating functions

_Probability & Statistics_

**One function that secretly stores every moment of a distribution. Differentiate at zero to read them off.**

The mean and variance are "moments" — averages of powers of $X$.

     The moment generating function (MGF) packs all of them into a single function $M_X(t)$.

---

This notebook builds the MGF idea up one step at a time. Run each cell, read the note above it, and watch the moments fall straight out of $M_X(t)$ — then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy + SciPy

### Step 1 — Pick a distribution and write down its MGF

We work with an **Exponential** distribution with rate $\lambda = 1.5$. Its moment generating function has a tidy closed form, $M_X(t) = \lambda / (\lambda - t)$, valid for $t < \lambda$. We define it as a Python function so we can probe it near $t = 0$.

In [ ]:
import numpy as np
from scipy.stats import expon

rng = np.random.default_rng(0)

lam = 1.5
M = lambda t: lam / (lam - t)   # MGF of Exponential(rate=lam)

### Step 2 — Read moments off the MGF by differentiating at zero

The magic of the MGF: $M'(0) = E[X]$ and $M''(0) = E[X^2]$. We approximate those two derivatives with finite differences using a tiny step `h`. Once we have the first two moments, the variance follows from $\operatorname{Var}(X) = E[X^2] - (E[X])^2$.

In [ ]:
# Finite-difference step for approximating the derivatives at t = 0.
h = 1e-4

m1 = (M(h) - M(-h)) / (2 * h)              # M'(0)  = E[X]
m2 = (M(h) - 2 * M(0) + M(-h)) / (h * h)   # M''(0) = E[X^2]
var_mgf = m2 - m1 ** 2                      # Var(X) = E[X^2] - (E[X])^2

print("MGF  E[X] =", round(m1, 4), " Var =", round(var_mgf, 4))
print("exact     E[X] = 1/lam =", round(1 / lam, 4), " Var = 1/lam^2 =", round(1 / lam ** 2, 4))

### Step 3 — Sanity-check against a Monte-Carlo sample

The MGF derivatives should agree with a brute-force simulation. We draw two million exponential samples and compare their sample mean and variance to the values we read off $M(t)$. They should line up to a few decimals.

In [ ]:
# Monte-Carlo check from a simulated exponential sample.
x = expon(scale=1 / lam).rvs(2_000_000, random_state=rng)

print("MC   E[X] =", round(x.mean(), 4), " Var =", round(x.var(), 4))

## Visualize the data & results

_Can you read the mean and E[X^2] off the MGF M(t) = lambda/(lambda - t) by differentiating at t = 0?_

### Step 1 — Plot the MGF curve

First we look at $M(t)$ itself across a range of $t$ values below the rate $\lambda = 1.5$. The curve climbs steeply and blows up as $t$ approaches $1.5$ — the MGF only exists for $t < \lambda$, which is exactly where the integral defining it converges.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# MGF of Exponential(rate=lam): M(t) = lam/(lam - t), valid for t < lam.
lam = 1.5
M = lambda t: lam / (lam - t)

# A spread of t values staying below the blow-up at t = lam = 1.5.
t = np.array([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.25])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(t, M(t), '-o', color='#4ea1ff', label='M(t)')
ax1.set_xlabel('t')
ax1.set_ylabel('M(t)')
ax1.legend()
ax1.set_title('MGF of Exponential rate 1.5: blows up at t = 1.5')

### Step 2 — Compare moments from the MGF, the exact formula, and simulation

Now we bar-chart the moments three ways: read off the MGF by differentiation, computed from the exact Exponential formulas ($E[X]=1/\lambda$, $E[X^2]=2/\lambda^2$), and estimated from a Monte-Carlo sample. All three should agree, confirming the MGF really does store the moments.

In [ ]:
# Read moments off M by finite-difference derivatives at t = 0.
h = 1e-4
m1 = (M(h) - M(-h)) / (2 * h)              # M'(0)  = E[X]
m2 = (M(h) - 2 * M(0) + M(-h)) / (h * h)   # M''(0) = E[X^2]
var_mgf = m2 - m1 ** 2

# Monte-Carlo sample for an independent estimate of the variance.
rng = np.random.default_rng(0)
xs = rng.exponential(1 / lam, 2_000_000)

# Compare MGF-derived, exact-formula, and simulated values side by side.
labels = ['E[X] MGF', 'E[X] exact', 'E[X^2] MGF', 'E[X^2] exact', 'Var MGF', 'Var MC']
vals = [m1, 1 / lam, m2, 2 / lam ** 2, var_mgf, xs.var()]
colors = ['#4ea1ff', '#7ee787', '#4ea1ff', '#7ee787', '#ffb454', '#c89bff']

ax2.bar(labels, vals, color=colors)
ax2.set_title('Moments of Exponential(1.5): MGF vs exact vs MC')
ax2.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()